***WORKSHOP - SCRAPER LES DONNEES DE PLUSIEURS PAGES***

Tuto 2 mais version adaptée à partir du site **books.toscrape.com** qui est un site fait exprès pour s'entrainer au scraping.

L'idée reste la même : récupérer les infos sur chaque page puis exporte en un JSON.

Je fais d'abord une version **avec regex** (on parse le HTML à la main), puis exactement le même exercice **sans regex**.

In [1]:
# On a juste besoin de requests pour télécharger les pages et re pour les regex.
import requests
import re
import html  # pour décoder les entités HTML genre &#39; -> '

# La pagination du site est du style page-1.html, page-2.html ... donc on garde l'url de base à part.
URL_BASE = "https://books.toscrape.com/catalogue/"

# Les notes sont écrites en lettres dans le HTML, on prépare une table de correspondance.
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

***VERSION 1 : AVEC REGEX***

In [2]:
# Première fonction : à partir du HTML d'une page, je récupère la liste des livres avec des regex.
# Chaque livre est dans un bloc <article class="product_pod"> ... </article>, donc je découpe d'abord par bloc.

def extraire_livres_regex(html_page):
    livres = []
    # re.S (DOTALL) pour que le .* prenne aussi les retours à la ligne.
    blocs = re.findall(r'<article class="product_pod">(.*?)</article>', html_page, re.S)

    for bloc in blocs:
        # le titre complet est dans l'attribut title du lien (h3 > a)
        titre = re.search(r'<h3><a href=".*?" title="(.*?)">', bloc).group(1)
        # petit piège des regex : le HTML contient des entités (&#39; pour ', &amp; pour &),
        # il faut les décoder à la main alors qu'un vrai parser le fait tout seul.
        titre = html.unescape(titre)
        # le prix, on attrape juste les chiffres après la £
        prix = re.search(r'price_color">£([\d.]+)', bloc).group(1)
        # la note est cachée dans la classe : star-rating Three -> Three
        note = re.search(r'star-rating (\w+)"', bloc).group(1)
        # la dispo (In stock) se trouve après la balise <i>
        dispo = re.search(r'instock availability">.*?</i>\s*(.*?)\s*</p>', bloc, re.S).group(1)

        livres.append({
            "titre": titre,
            "prix": float(prix),
            "note": NOTES[note],
            "dispo": dispo
        })

    return livres

In [3]:
# Deuxième fonction : trouver le lien vers la page suivante.
# Dans le HTML il y a <li class="next"><a href="page-2.html">next</a></li>.
# Si on ne trouve rien, c'est qu'on est sur la dernière page, donc on renvoie None.

def page_suivante_regex(html):
    suivant = re.search(r'<li class="next"><a href="(.*?)">next</a>', html)
    if suivant:
        return suivant.group(1)
    return None

In [4]:
# Maintenant la boucle principale : on part de la page 1 et on suit "next" tant qu'il y en a une.
# Je mets une limite de pages pour ne pas scraper les 50 pages à chaque test (à enlever pour tout récupérer).

def scraper_avec_regex(limite_pages=5):
    tous_les_livres = []
    url = "page-1.html"
    compteur = 0

    while url and compteur < limite_pages:
        reponse = requests.get(URL_BASE + url, timeout=20)
        reponse.encoding = "utf-8"  # sinon la £ s'affiche mal
        html = reponse.text

        livres = extraire_livres_regex(html)
        tous_les_livres += livres
        compteur += 1
        print(f"Page {compteur} scrapée ({url}) -> {len(livres)} livres récupérés.")

        url = page_suivante_regex(html)

    print(f"\nTerminé ! {len(tous_les_livres)} livres au total sur {compteur} pages.")
    return tous_les_livres

livres_regex = scraper_avec_regex(limite_pages=5)

Page 1 scrapée (page-1.html) -> 20 livres récupérés.
Page 2 scrapée (page-2.html) -> 20 livres récupérés.
Page 3 scrapée (page-3.html) -> 20 livres récupérés.
Page 4 scrapée (page-4.html) -> 20 livres récupérés.
Page 5 scrapée (page-5.html) -> 20 livres récupérés.

Terminé ! 100 livres au total sur 5 pages.


In [5]:
# On regarde les premiers résultats pour vérifier que tout est propre.
for livre in livres_regex[:5]:
    print(f"{livre['titre']} | {livre['prix']}£ | note {livre['note']}/5 | {livre['dispo']}")

A Light in the Attic | 51.77£ | note 3/5 | In stock
Tipping the Velvet | 53.74£ | note 1/5 | In stock
Soumission | 50.1£ | note 1/5 | In stock
Sharp Objects | 47.82£ | note 4/5 | In stock
Sapiens: A Brief History of Humankind | 54.23£ | note 5/5 | In stock


***VERSION 2 : SANS REGEX (sélecteurs CSS avec parsel)***

Là on refait exactement la même chose mais sans aucune regex. On utilise `Selector` de **parsel** (la lib que Scrapy utilise en interne) pour naviguer dans le HTML avec des sélecteurs CSS. C'est beaucoup plus lisible et ça casse moins facilement que des regex.

In [6]:
from parsel import Selector

# Même logique : une fonction qui extrait les livres d'une page, mais avec des sélecteurs CSS.

def extraire_livres_css(html):
    livres = []
    selecteur = Selector(text=html)

    for article in selecteur.css("article.product_pod"):
        titre = article.css("h3 a::attr(title)").get()
        # ::text récupère le texte de la balise, on enlève la £ et on garde les chiffres
        prix = article.css("p.price_color::text").get().replace("£", "")
        # la classe complète est "star-rating Three", on prend juste le 2e mot
        classe_note = article.css("p.star-rating::attr(class)").get()
        note = classe_note.split()[1]
        dispo = " ".join(article.css("p.instock.availability::text").getall()).strip()

        livres.append({
            "titre": titre,
            "prix": float(prix),
            "note": NOTES[note],
            "dispo": dispo
        })

    return livres

In [7]:
# La page suivante en CSS : on cible directement le lien dans le <li class="next">.

def page_suivante_css(html):
    selecteur = Selector(text=html)
    return selecteur.css("li.next a::attr(href)").get()  # renvoie None tout seul si pas trouvé

In [8]:
# La boucle principale est strictement la même, on change juste les deux fonctions appelées.

def scraper_sans_regex(limite_pages=5):
    tous_les_livres = []
    url = "page-1.html"
    compteur = 0

    while url and compteur < limite_pages:
        reponse = requests.get(URL_BASE + url, timeout=20)
        reponse.encoding = "utf-8"
        html = reponse.text

        livres = extraire_livres_css(html)
        tous_les_livres += livres
        compteur += 1
        print(f"Page {compteur} scrapée ({url}) -> {len(livres)} livres récupérés.")

        url = page_suivante_css(html)

    print(f"\nTerminé ! {len(tous_les_livres)} livres au total sur {compteur} pages.")
    return tous_les_livres

livres_css = scraper_sans_regex(limite_pages=5)

Page 1 scrapée (page-1.html) -> 20 livres récupérés.
Page 2 scrapée (page-2.html) -> 20 livres récupérés.
Page 3 scrapée (page-3.html) -> 20 livres récupérés.
Page 4 scrapée (page-4.html) -> 20 livres récupérés.
Page 5 scrapée (page-5.html) -> 20 livres récupérés.

Terminé ! 100 livres au total sur 5 pages.


In [9]:
# On vérifie que les deux méthodes donnent bien le même résultat.
for livre in livres_css[:5]:
    print(f"{livre['titre']} | {livre['prix']}£ | note {livre['note']}/5 | {livre['dispo']}")

print("\nMêmes données avec et sans regex ?", livres_regex == livres_css)

A Light in the Attic | 51.77£ | note 3/5 | In stock
Tipping the Velvet | 53.74£ | note 1/5 | In stock
Soumission | 50.1£ | note 1/5 | In stock
Sharp Objects | 47.82£ | note 4/5 | In stock
Sapiens: A Brief History of Humankind | 54.23£ | note 5/5 | In stock

Mêmes données avec et sans regex ? True


# BONUS : un petit récap des données récupérées

In [10]:
# Quelques stats simples pour finir, sans bibliothèque supplémentaire.
prix = [livre["prix"] for livre in livres_css]

print(f"Nombre de livres récupérés : {len(livres_css)}")
print(f"Prix moyen : {sum(prix)/len(prix):.2f}£")
print(f"Livre le moins cher : {min(prix)}£")
print(f"Livre le plus cher : {max(prix)}£")

# le mieux noté
meilleur = max(livres_css, key=lambda l: l["note"])
print(f"\nUn des livres les mieux notés : {meilleur['titre']} ({meilleur['note']}/5)")

Nombre de livres récupérés : 100
Prix moyen : 34.56£
Livre le moins cher : 10.16£
Livre le plus cher : 58.11£

Un des livres les mieux notés : Sapiens: A Brief History of Humankind (5/5)


In [11]:
# On sauvegarde le résultat dans un fichier JSON au cas où on veuille réutiliser les données.
import json

with open("livres.json", "w", encoding="utf-8") as f:
    json.dump(livres_css, f, ensure_ascii=False, indent=2)

print("Fichier livres.json enregistré.")

Fichier livres.json enregistré.
